**Tidy3D** es un motor de simulación electromagnética especializado en fotónica que resuelve numéricamente cómo se propaga la luz en estructuras complejas, utilizando métodos como FDTD para aproximar las ecuaciones de Maxwell. Funciona principalmente en la nube, lo que permite ejecutar simulaciones intensivas sin depender del hardware local. A partir de un diseño, puede calcular propiedades como transmisión, reflexión, pérdidas y distribución de campos, siendo la herramienta encargada de darle significado físico a los dispositivos fotónicos.

**PhotonForge** es un entorno de diseño programático para circuitos fotónicos que permite construir, organizar y manipular dispositivos como guías de onda, resonadores o interferómetros. Su enfoque está en definir la geometría, las conexiones y la estructura del sistema de manera flexible usando código, facilitando la automatización y exploración de diseños. Además, sirve como intermediario entre el diseño conceptual y la simulación física, integrándose con motores externos para evaluar el comportamiento de los dispositivos.

**SiEPIC Tools** es un conjunto de herramientas y librerías que actúa como un PDK (Process Design Kit) para fotónica en silicio, proporcionando componentes predefinidos, parámetros tecnológicos y reglas de fabricación basadas en procesos reales. Esto permite diseñar dispositivos que no solo son teóricamente válidos, sino también fabricables en entornos industriales. Incluye modelos de dispositivos y configuraciones que reflejan condiciones físicas realistas.

La relación entre **Tidy3D y PhotonForge** es de dependencia funcional: PhotonForge se encarga de construir y organizar el diseño del circuito fotónico, pero delega completamente el cálculo físico a Tidy3D. Cuando se requiere simular un dispositivo, PhotonForge traduce la geometría y los parámetros definidos en un problema que Tidy3D puede resolver, y luego utiliza los resultados para analizar el comportamiento del sistema. En este sentido, PhotonForge actúa como la capa de diseño y Tidy3D como el motor de simulación que ejecuta los cálculos complejos.

La relación entre **SiEPIC y PhotonForge** es de integración de conocimiento tecnológico: SiEPIC proporciona el conjunto de componentes, materiales y reglas que representan un proceso real de fabricación, mientras que PhotonForge utiliza esa información para construir diseños más realistas y consistentes. Al establecer SiEPIC como tecnología por defecto, PhotonForge deja de trabajar con modelos genéricos y pasa a usar elementos validados, lo que permite que los diseños sean más cercanos a aplicaciones prácticas en la industria de la fotónica.


```
pip install photonforge
```

```
pip install tidy3d
```

[Link para obtener la API KEY](https://my.simulation.cloud/login?ref=https://tidy3d.simulation.cloud)
```
tidy3d configure --apikey=XXX
```
> **nota**: si se instaló con el archivo `requirements.txt` no es necesario ejecutar los comandos `pip install`>


ahora podemos visualizar la estructura con klayuot (normalmente encontrado en los repositorios de buena parte de las distribuciones más famosas de linux) `sudo pacman -S klayout`

# Diseño de un Interferómetro Mach-Zehnder (MZI) en Photonforge

Este notebook documenta el proceso de diseño de un **Inteferómetro Mach-Zehnder (MZI)**.
Un MZI es un dispositivo fundamental en fotónica que divide la luz en dos caminos (baroz), genera un desfase en uno de ellos y luego los recombina para producir interferencia.

## Configuración del Entorno y Tecnología

En esta sección definimos las propiedades fisicas de los materiales. Photonforge utilza un objeto `Technology` para saber qué grosor tiene las capas y qué índices de refracción aplicar en simulaciones posteriores

In [1]:
import photonforge as pf
import numpy as np
import tidy3d as td

# --- Configuración de Tecnología ---
# Creamos una tecnología base que simula un proceso de fabricación estándar de silicio sobre aislante (SOI).
# Definimos:
# - core_thickness (0.22 µm): El grosor estándar de la capa de silicio.
# - strip_width (0.5 µm): El ancho nominal de las guías de onda monomodo.
tech = pf.basic_technology(
    core_thickness=0.22,
    strip_width=0.5,
    clad_thickness=1.0,  # Recubrimiento superior (SiO2)
    box_thickness=1.0,   # Óxido enterrado (SiO2)
)
# Establecemos esta tecnología como la predeterminada para todos los componentes.
pf.config.default_technology = tech

# --- Parámetros de Diseño Global ---
w_wg = 0.5         # Ancho de la guía de onda en micras.
layer_wg = (1, 0)  # Capa GDS donde se dibujará el núcleo de silicio.

# --- Especificación de Puerto (PortSpec) ---
# El PortSpec define el perfil transversal de la guía de onda en el puerto.
# Es crucial para que Photonforge sepa cómo conectar componentes lógicamente.
port_spec = pf.PortSpec("Puerto silicio", w_wg, (0.0, 0.22), 1)

In [2]:
import photonforge as pf
help(pf.PortSpec)

Help on class PortSpec in module photonforge:

class PortSpec(builtins.object)
 |  PortSpec(description, width, limits, num_modes=1, polarization=None, target_neff=1, path_profiles=(), added_solver_modes=0, voltage_path=None, current_path=None, default_radius=0)
 |
 |  Optical or electrical port specification.
 |
 |  This class is mainly used to create optical or electrical waveguide
 |  templates for a :class:`Technology`.
 |
 |  Args:
 |      description (str): Description of the port specification.
 |      width (float): In-plane dimension of the port.
 |      limits (Sequence[float]): Out-of-plane bounds of the port.
 |      num_modes (int): Number of modes supported by this port.
 |      polarization (Literal["TE", "TM", ""] | None): Mode polarization
 |        selection.
 |      target_neff (float): Target effective index for numerical mode
 |        solving.
 |      path_profiles (Iterable[tuple[float, float, str | tuple[int, int]]] | dict[str, tuple[float, float, str | tuple[in

## Diseño del Divisor de Potencia (Y-Splitter)

El **Y-Splitter** divide la potencia de entrada (100%) en dos salidas de igual potencia (50% cada una). Utilizamos un diseño optimizado con un "tip" central para minimiar las pérdidas por reflexión.

In [3]:
# --- Parámetros Geométricos del Splitter ---
w_tip = 0.1     # Ancho de la punta central (ayuda a la transición suave de la luz).
L = 3.0         # Longitud de la zona de transición.
gap = 0.1       # Separación inicial entre las dos ramas de salida.
s_len = 5.0     # Longitud de la curva en S (S-bend) para separar los brazos.
s_offset = 1.5  # Distancia vertical final desde el centro hasta cada brazo.

half_w = w_wg / 2
half_g = gap / 2

# --- Construcción del Cuerpo (Polígono) ---
# Definimos los vértices que forman la "Y" central donde la guía se ensancha y divide.
ysplitter_pts = [
    (0.0, -half_w), (0.0, half_w),             # Entrada
    (L, half_g + w_wg), (L, half_g),           # Salida superior
    (L*0.6, w_tip / 2), (L*0.6, -w_tip / 2),   # Punto de división (Tip)
    (L, -half_g), (L, -(half_g + w_wg)),       # Salida inferior
]
splitter_polygon = pf.Polygon(ysplitter_pts)

# --- Guías de Acceso (Paths) ---
# Creamos las rutas que conectan el cuerpo del splitter con el resto del circuito.
# path_top: Sale del cuerpo, hace una curva en S hacia arriba y termina en un segmento recto.
path_top = pf.Path((L, half_g + half_w), w_wg).s_bend((L + s_len, s_offset)).segment((L + s_len + 2.0, s_offset))
# path_bot: Simétrico al anterior, pero hacia abajo.
path_bot = pf.Path((L, -(half_g + half_w)), w_wg).s_bend((L + s_len, -s_offset)).segment((L + s_len + 2.0, -s_offset))
# path_in: Guía de entrada recta.
path_in = pf.Path((-2.0, 0.0), w_wg).segment((0.0, 0.0))

# --- Creación del Componente Splitter ---
ysplitter = pf.Component("Y_Splitter")
ysplitter.add(layer_wg, splitter_polygon, path_top, path_bot, path_in)

# --- Definición de Puertos ---
# P_IN: Donde entra la luz (dirección 0°).
ysplitter.add_port(pf.Port((-2.0, 0.0), 0.0, port_spec), "P_IN")
# P_OUT_TOP/BOT: Donde sale la luz (dirección 180° para indicar conexión hacia afuera).
ysplitter.add_port(pf.Port((L + s_len + 2.0, s_offset), 180.0, port_spec), "P_OUT_TOP")
ysplitter.add_port(pf.Port((L + s_len + 2.0, -s_offset), 180.0, port_spec), "P_OUT_BOT")

'P_OUT_BOT'

In [4]:
pf.write_layout("ysplitter.gds", ysplitter)

## Brazos del Interferómetro
Aquí definimos los dos caminso. Para que haya interferencia, uno de los brazos debe tener una longitud física distinta al otro (desfase geométrico)

In [7]:
# --- Parámetros de los Brazos ---
L_short = 80.0  # Longitud base del interferómetro.
h_extra = 15.0  # Altura de la "serpentina" para el brazo largo.

# --- Brazo Corto (Referencia) ---
# Una guía de onda recta que une la entrada con la salida.
short_arm = pf.Component("short_arm")
path_short = pf.Path((0.0, 0.0), w_wg).segment((L_short, 0.0))
short_arm.add(layer_wg, path_short)
short_arm.add_port(pf.Port((0.0, 0.0), 0.0, port_spec), "P1")
short_arm.add_port(pf.Port((L_short, 0.0), 180.0, port_spec), "P2")

# --- Brazo Largo (Retraso de Fase) ---
# Creamos una serpentina. Aunque empieza y termina en la misma coordenada Y que el corto,
# el recorrido total es mayor, lo que retrasa la fase de la luz.
long_arm = pf.Component("long_arm")
path_long = (
    pf.Path((0.0, 0.0), w_wg)
    .segment((10.0, 0.0))                 # Tramo inicial
    .segment((10.0, -h_extra))            # Bajada
    .segment((L_short - 10.0, -h_extra))  # Tramo largo inferior
    .segment((L_short - 10.0, 0.0))       # Subida
    .segment((L_short, 0.0))              # Tramo final
)
long_arm.add(layer_wg, path_long)
long_arm.add_port(pf.Port((0.0, 0.0), 0.0, port_spec), "P1")
long_arm.add_port(pf.Port((L_short, 0.0), 180.0, port_spec), "P2")

'P2'

In [8]:
pf.write_layout("long_arm.gds", long_arm)

## Ensamblaje Final del MZI

En el último paso, unimos todas las piezas. Utilizamos el primer splitter para dividir la luz y una copai rota del mismo para recombinarla.

In [9]:
# --- Creación del Componente Principal ---
mzi = pf.Component("MZI")

# 1. Colocamos el Splitter de Entrada
ref_in = mzi.add_reference(ysplitter)

# 2. Conectamos los Brazos a las salidas del Splitter de Entrada
ref_s = mzi.add_reference(short_arm)
ref_s.connect("P1", ref_in.component.ports["P_OUT_TOP"])

ref_l = mzi.add_reference(long_arm)
ref_l.connect("P1", ref_in.component.ports["P_OUT_BOT"])

# 3. Colocamos el Recombinador (Splitter de Salida)
# El recombinador es el mismo Y-Splitter pero rotado 180 grados.
ref_out = mzi.add_reference(ysplitter)
ref_out.rotate(180)

# --- Cálculo de Posicionamiento Matemático ---
# Para evitar "gaps" o huecos entre polígonos, calculamos la posición exacta
# del recombinador basándonos en el final de los brazos.
centro_x = ref_s.origin[0] + L_short
centro_y = 0.0 # El eje central del dispositivo.

# La posición del origen del recombinador se desplaza su propia longitud 
# hacia la derecha para que sus "brazos" toquen exactamente el final de nuestros brazos.
ref_out.origin = (centro_x + (L + s_len + 2.0), centro_y)

# 4. Puertos Globales
# Definimos los puertos de entrada y salida de todo el circuito MZI.
mzi.add_port(pf.Port(ref_in.component.ports["P_IN"].center, 0.0, port_spec), "IN")
mzi.add_port(pf.Port(ref_out.component.ports["P_IN"].center, 180.0, port_spec), "OUT")

# --- Exportación ---
# Guardamos el diseño en formato GDSII, compatible con software de simulación y fabricación.
pf.write_layout("mzi_matematico.gds", mzi)

In [10]:
help(pf.write_layout)

Help on built-in function write_layout in module photonforge.extension:

write_layout(filename, *components, paths_to_polygons=True, compression_level=9, fracture_limit=0, library_name='LIBRARY', library_properties=None, pre_export_function=None)
    Export this component as a GDSII or OASIS file.

    The layout format is defined by the file name extension: ".gds" or
    ".oas" for GDSII or OASIS files, respectively.

    Args:
        filename (str): Output file name.
        *components (Component): Components to write to the layout file.
        paths_to_polygons (bool): If ``True``, all :class:`Path` structures
          will be stored as polygonal contours.
        compression_level (int): Level of compression of the file. Disabled
          if set to 0. Maximal compression level is 9. *(OASIS only)*
        fracture_limit (int): Polygons with a number of vertices above this
          limit will be fractured into multiple polygons. The limit is
          ignored if not above 4. *

## ¿Qué estás logrando físicamente?

La diferencia clave es:

$$
\Delta L = L_{\text{largo}} - L_{\text{corto}}
$$

Esto produce un desfase:

$$
\Delta \phi = \frac{2\pi n_{\text{eff}}}{\lambda} \Delta L
$$

👉 Ese desfase es lo que genera:

* interferencia constructiva
* interferencia destructiva

👉 Resultado: comportamiento tipo **filtro espectral**